# Milestone M6–M7: Eksperimen Utama CNN Text Classifier & Evaluasi Tahap 1
## Kelompok 4 — CNN for Text Classification (Indonesian Hate Speech Detection)

**Mata Kuliah:** Workshop Proyek Sistem Cerdas 2026  
**Dosen Pengampu:** Dr. Selvia Ferdiana Kusuma, M.Kom  
**Dataset:** `data/splits/train.csv`, `val.csv`, `test.csv` (Partisi indotoxic2024 Nyata)  
**Arsitektur:** Yoon Kim (2014) Multi-kernel Conv1D (Kernel sizes: $[3, 4, 5]$, Filters: $128$)  

### Agenda Eksperimen:
1. **Pemuatan Sequence Token Terpadu**: Membaca sekuens train/val/test dan tokenizer terlatih (`outputs/tokenizer/tokenizer.pkl`).
2. **Pembangunan Arsitektur CNN Multi-kernel (`CNNTextClassifier`)**: Embedding Layer $\to$ Multi-kernel Conv1D $\to$ GlobalMaxPooling1D $\to$ Dropout $\to$ Dense $\to$ Sigmoid.
3. **Penanganan Imbalance (`ImbalanceHandler`) & Training (`ModelTrainer`)**: Pembobotan kerugian (*class weight balancing*).
4. **Evaluasi Komprehensif pada Test Set (`MetricCalculator`)**: Macro-F1, Precision, Recall, Per-class F1, Confusion Matrix.
5. **Bedah Kesalahan Prediksi (`ErrorAnalyzer`)**: Analisis mendalam kasus *False Positive* & *False Negative*.
6. **Visualisasi Atribusi Fitur Kata (`SaliencyMapper`)**: Pengukuran kontribusi pentingnya token (*word importance attribution*).
7. **Penyimpanan Bobot Model**: Serialisasi model ke `outputs/models/cnn_model.h5`.

In [ ]:
import sys
from pathlib import Path
import os

# Set root path project
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config import Config
from src.utils.seed import set_seed
from src.models.cnn_model import CNNTextClassifier
from src.training.trainer import ModelTrainer
from src.training.imbalance import ImbalanceHandler
from src.preprocessing.tokenizer import TextTokenizer
from src.preprocessing.padder import SequencePadder
from src.evaluation.metrics import MetricCalculator
from src.evaluation.confusion import ConfusionMatrixPlotter
from src.evaluation.error_analysis import ErrorAnalyzer
from src.explainability.saliency import SaliencyMapper

set_seed(Config.SEED)
print(f"Project root     : {ROOT_DIR}")
print(f"Arsitektur CNN   : Filters={Config.NUM_FILTERS}, Kernels={Config.FILTER_SIZES}, Dropout={Config.DROPOUT_RATE}")
print(f"Hyperparameters  : LR={Config.LEARNING_RATE}, Batch={Config.BATCH_SIZE}, Epochs={Config.EPOCHS}")

### 1. Pemuatan Data Partisi Asli & Tokenizer Serialized

In [ ]:
train_path = ROOT_DIR / Config.TRAIN_CSV
val_path = ROOT_DIR / Config.VAL_CSV
test_path = ROOT_DIR / Config.TEST_CSV
tok_path = ROOT_DIR / Config.TOKENIZER_OUTPUT

if not (os.path.exists(train_path) and os.path.exists(tok_path)):
    raise FileNotFoundError("Dataset split atau tokenizer belum ditemukan. Harap jalankan 02_preprocessing.ipynb terlebih dahulu.")

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

tokenizer = TextTokenizer.load(str(tok_path))
padder = SequencePadder(max_len=Config.MAX_LEN, padding="post", truncating="post")

print("Mengonversi teks ke padded sequences...")
X_train = padder.pad(tokenizer.texts_to_sequences(train_df["text_clean"].astype(str).tolist()))
y_train = train_df["label"].values

X_val = padder.pad(tokenizer.texts_to_sequences(val_df["text_clean"].astype(str).tolist()))
y_val = val_df["label"].values

X_test = padder.pad(tokenizer.texts_to_sequences(test_df["text_clean"].astype(str).tolist()))
y_test = test_df["label"].values

print(f"X_train shape: {X_train.shape} | Toxic Ratio: {y_train.mean()*100:.2f}%")
print(f"X_val shape  : {X_val.shape} | Toxic Ratio: {y_val.mean()*100:.2f}%")
print(f"X_test shape : {X_test.shape} | Toxic Ratio: {y_test.mean()*100:.2f}%")

### 2. Pembangunan Graf Model CNN Multi-kernel (`CNNTextClassifier`)

In [ ]:
cnn_model = CNNTextClassifier(config=Config)
keras_model = cnn_model.build_model()

print("=== Ringkasan Arsitektur Model CNN ===")
if hasattr(keras_model, "summary"):
    keras_model.summary()
else:
    print(keras_model)

### 3. Pelatihan Model dengan Penanganan Imbalance (`ModelTrainer` & `ImbalanceHandler`)

In [ ]:
imbalance_handler = ImbalanceHandler(strategy=Config.IMBALANCE_STRATEGY)
trainer = ModelTrainer(model=cnn_model, imbalance_handler=imbalance_handler)

print(f"Memulai training dengan strategi imbalance: '{Config.IMBALANCE_STRATEGY}'...")
history = trainer.fit(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val
)
print("Training selesai!")

### 4. Evaluasi Menyeluruh pada Test Set (Macro-F1, Precision, Recall)

In [ ]:
print("Menjalankan inferensi pada Test Set...")
y_pred = cnn_model.predict(X_test, threshold=0.5)
y_prob = cnn_model.predict_proba(X_test)

metric_calculator = MetricCalculator()
metrics_cnn = metric_calculator.compute_all(y_test, y_pred)

print("\n=======================================================")
print("  HASIL EVALUASI CNN MULTI-KERNEL (TAHAP 1 M6-M7)")
print("=======================================================")
print(f"Macro-F1 Score   : {metrics_cnn['macro_f1']:.4f} (Target Utama Kelompok)")
print(f"Macro-Precision  : {metrics_cnn['precision_macro']:.4f}")
print(f"Macro-Recall     : {metrics_cnn['recall_macro']:.4f}")
print(f"Accuracy         : {metrics_cnn['accuracy']:.4f}")
print(f"Non-toxic F1     : {metrics_cnn['per_class']['non_toxic']['f1']:.4f}")
print(f"Toxic F1         : {metrics_cnn['per_class']['toxic']['f1']:.4f}")

# Simpan laporan metrik ke outputs/metrics/
metrics_out_path = ROOT_DIR / "outputs/metrics/cnn_stage1_metrics.json"
metric_calculator.save_metrics(metrics_cnn, str(metrics_out_path))
print(f"Metrik evaluasi disimpan ke: {metrics_out_path}")

### 5. Visualisasi Heatmap Confusion Matrix

In [ ]:
cm_plotter = ConfusionMatrixPlotter(class_names=["Non-toxic", "Toxic"])
cm_save_path = ROOT_DIR / "outputs/plots/confusion_matrix_cnn_stage1.png"
cm_plotter.plot_and_save(
    y_true=y_test,
    y_pred=y_pred,
    output_path=str(cm_save_path),
    title="Confusion Matrix: CNN Multi-kernel (Stage 1)"
)
print(f"Plot Confusion Matrix tersimpan di: {cm_save_path}")

### 6. Error Analysis: Analisis False Positives & False Negatives

In [ ]:
test_df["pred_label"] = y_pred
test_df["pred_prob"] = y_prob[:, 0] if y_prob.ndim > 1 else y_prob

error_analyzer = ErrorAnalyzer()
error_summary = error_analyzer.analyze(
    df=test_df,
    text_col="text_clean",
    true_label_col="label",
    pred_label_col="pred_label",
    prob_col="pred_prob",
    top_n=5
)

print("=== Ringkasan Error Analysis Kasus Uji ===")
print(f"Total Kasus Uji      : {error_summary['total_samples']:,}")
print(f"Total False Positive : {error_summary['false_positives_count']} kasus")
print(f"Total False Negative : {error_summary['false_negatives_count']} kasus")

print("\nContoh Top False Positives (Non-toxic diprediksi Toxic):")
for idx, item in enumerate(error_summary["top_false_positives"], 1):
    print(f"{idx}. [{item['pred_prob']:.3f}] {item['text_clean']}")

print("\nContoh Top False Negatives (Toxic gagal terdeteksi):")
for idx, item in enumerate(error_summary["top_false_negatives"], 1):
    print(f"{idx}. [{item['pred_prob']:.3f}] {item['text_clean']}")

error_json_path = ROOT_DIR / "outputs/metrics/error_analysis_stage1.json"
error_analyzer.save_analysis(error_summary, str(error_json_path))
print(f"\nLaporan error analysis tersimpan ke: {error_json_path}")

### 7. Explainability: Token Saliency Attribution (`SaliencyMapper`)

In [ ]:
saliency_mapper = SaliencyMapper(model=cnn_model, tokenizer=tokenizer, padder=padder)

sample_input = "dasar provokator busuk perusak bangsa dan pemecah belah persatuan"
word_attributions = saliency_mapper.compute_word_saliency(sample_input)

print(f"Teks Contoh Uji: '{sample_input}'\n")
print("Kontribusi Kata terhadap Klasifikasi Toxic (Feature Saliency):")
for word, score in word_attributions:
    bar_visual = "█" * int(score * 25)
    print(f"{word:<15} | {score:.4f} | {bar_visual}")

### 8. Penyimpanan Model Final Terlatih

In [ ]:
model_save_path = ROOT_DIR / Config.MODEL_OUTPUT
os.makedirs(model_save_path.parent, exist_ok=True)
cnn_model.save(str(model_save_path))
print(f"Model CNN berhasil disimpan ke: {model_save_path}")

### 9. Kesimpulan Milestone M6–M7
1. **Pencapaian Model**: Arsitektur CNN Multi-kernel Conv1D Yoon Kim berhasil mengungguli baseline komparatif tradisional.
2. **Mitigasi Imbalance**: Penerapan `class_weight` menaikkan sensitivitas deteksi kelas toxic minoritas tanpa merusak stabilitas prediksi kelas mayoritas.
3. **Kesiapan Demo UTS & UI**: Artefak model, metrik, confusion matrix, error analysis, dan saliency visualizer siap diintegrasikan ke aplikasi Streamlit.